## Predicción de ventas anuales de videojuegos en función del mes de lanzamiento
#### Guillem Dominguez, Sofia Gallardo, Carlos Pamplona

In [2]:
# Librerías de tratamiento de datos
import numpy as np
import pandas as pd

# Librerías para visualización
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

# Librerías de Machine Learning
import sklearn

In [3]:
#FUNCIONES

def parse_estimated_owners(owner_val):
    """Convierte la variable "estimated_owners" de string a tupla de enteros.

    Parámetros:
        owner_val (any): String con el rango de "estimated_owners".

    Retorna:
        tupla: Dos enteros que marcan el rango de "estimated_owners".
    """
    # Si ya es una tupla con 2 elementos de tipo numérico, la conservamos tal cual
    if isinstance(owner_val, tuple) and len(owner_val) == 2:
        if isinstance(owner_val[0], (int, float)) and isinstance(owner_val[1], (int, float)):
            return (int(owner_val[0]), int(owner_val[1]))

    # Si es nulo o no es una cadena de texto, devolvemos un rango por defecto (0, 0)
    if pd.isna(owner_val) or not isinstance(owner_val, str):
        return (0, 0)

    try:
        # Limpiamos y dividimos la cadena por el guion '-'
        parts = owner_val.split('-')
        if len(parts) == 2:
            low = int(parts[0].strip().replace(',', ''))
            high = int(parts[1].strip().replace(',', ''))
            return (low, high)
    except Exception:
        pass

    return (0, 0)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


El código inferior es para parsear la base de datos en formato JSON y convertirla a CSV para una mejor carga.

In [4]:
# # !pip install ijson
# import ijson

# file_path = "/content/drive/MyDrive/TFM_DS_EquipoA/Colab/Datasets/games.json"
# output_path = "/content/drive/MyDrive/TFM_DS_EquipoA/Colab/Datasets/steam_games.csv"

# chunk_size = 5000

# # Primera pasada para identificar columnas
# all_columns = {"steam_appid"}
# batch = []

# with open(file_path, "rb") as f:

#     for game_id, game_data in ijson.kvitems(f, ""):

#         game_data["steam_appid"] = game_id
#         batch.append(game_data)

#         if len(batch) >= chunk_size:
#             df = pd.json_normalize(batch)
#             all_columns.update(df.columns)
#             batch = []

#     if batch:
#         df = pd.json_normalize(batch)
#         all_columns.update(df.columns)

# # Fijar el orden de las columnas
# columns = ["steam_appid"] + sorted(
#     all_columns - {"steam_appid"}
# )

# # Segunda pasada para generar el CSV
# batch = []
# first_chunk = True

# with open(file_path, "rb") as f:

#     for game_id, game_data in ijson.kvitems(f, ""):

#         game_data["steam_appid"] = game_id
#         batch.append(game_data)

#         if len(batch) >= chunk_size:

#             df = pd.json_normalize(batch)

#             # Asegurar las mismas columnas y orden
#             df = df.reindex(columns=columns)

#             df.to_csv(
#                 output_path,
#                 mode="w" if first_chunk else "a",
#                 header=first_chunk,
#                 index=False,
#                 encoding="utf-8"
#             )

#             first_chunk = False
#             batch = []

#     if batch:
#         df = pd.json_normalize(batch)
#         df = df.reindex(columns=columns)

#         df.to_csv(
#             output_path,
#             mode="w" if first_chunk else "a",
#             header=first_chunk,
#             index=False,
#             encoding="utf-8"
#         )


In [5]:
csv_path = "/content/drive/MyDrive/TFM_DS_EquipoA/Colab/Datasets/steam_games.csv"

# Cargamos el dataset utilizando el motor 'python' y configurando 'on_bad_lines' para ignorar o procesar errores de parsing
steam_data = pd.read_csv(csv_path, engine='python', on_bad_lines='skip')

In [6]:
steam_data.head()

,steam_appid,about_the_game,achievements,average_playtime_2weeks,average_playtime_forever,categories,detailed_description,developers,discount,dlc_count,...,tags.Wholesome,tags.Word Game,tags.World War I,tags.World War II,tags.Wrestling,tags.Zombies,tags.e-sports,user_score,website,windows
0,2539430,NaN,0,0,0,[],NaN,[],0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,True
1,496350,"Springtime, April: when the cherry trees come ...",0,0,8,"['Single-player', 'Steam Trading Cards', 'Stea...","Springtime, April: when the cherry trees come ...",['minori'],65,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,http://mangagamer.org/supipara,True
2,1034400,"Immerse yourself in the most beloved, mystical...",0,0,0,"['Single-player', 'Family Sharing']","Immerse yourself in the most beloved, mystical...",['Somer Games'],0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,https://www.facebook.com/8FloorGames/,True
3,3292190,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",19,0,0,"['Single-player', 'Steam Achievements', 'Famil...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['유진게임즈'],0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,True
4,3631080,Its not just a Maze; its a Quest! Enter the ca...,0,0,0,"['Single-player', 'VR Only', 'Steam Leaderboar...",Its not just a Maze; its a Quest! Enter the ca...,['Reality Expanded LLC'],0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,https://www.realityexpanded.com/books-games,True


In [7]:
steam_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35860 entries, 0 to 35859
Columns: 495 entries, steam_appid to windows
dtypes: bool(3), float64(454), int64(15), object(23)
memory usage: 134.7+ MB


# Preparación de los datos

In [8]:
# Eliminamos columnas que no tienen información de interés

to_drop = ['header_image', 'website', 'support_url', 'support_email', 'metacritic_url', 'windows', 'mac', 'linux', 'screenshots', 'movies', 'score_rank']

steam_data_prepared = steam_data.drop(to_drop, axis=1)

In [9]:
steam_data_prepared.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35860 entries, 0 to 35859
Columns: 484 entries, steam_appid to user_score
dtypes: float64(453), int64(15), object(16)
memory usage: 132.4+ MB


In [10]:
# Hacemos cambios de tipo de datos

steam_data_prepared['name'] = steam_data_prepared['name'].astype('string')
steam_data_prepared['release_date'] = pd.to_datetime(steam_data_prepared['release_date'])
steam_data_prepared['required_age'] = steam_data_prepared['required_age'].astype('int32')
steam_data_prepared['price'] = steam_data_prepared['price'].astype('float32')
steam_data_prepared['dlc_count'] = steam_data_prepared['dlc_count'].astype('int32')
steam_data_prepared['detailed_description'] = steam_data_prepared['detailed_description'].astype('string')
steam_data_prepared['about_the_game'] = steam_data_prepared['about_the_game'].astype('string')
steam_data_prepared['short_description'] = steam_data_prepared['short_description'].astype('string')
steam_data_prepared['reviews'] = steam_data_prepared['reviews'].astype('string')
steam_data_prepared['metacritic_score'] = steam_data_prepared['metacritic_score'].astype('int32')
steam_data_prepared['achievements'] = steam_data_prepared['achievements'].astype('int32')
steam_data_prepared['recommendations'] = steam_data_prepared['recommendations'].astype('int32')
steam_data_prepared['notes'] = steam_data_prepared['notes'].astype('string')
steam_data_prepared['developers'] = steam_data_prepared['developers'].astype('string')
steam_data_prepared['publishers'] = steam_data_prepared['publishers'].astype('string')
steam_data_prepared['categories'] = steam_data_prepared['categories'].astype('string')
steam_data_prepared['genres'] = steam_data_prepared['genres'].astype('string')
steam_data_prepared['user_score'] = steam_data_prepared['user_score'].astype('int32')
steam_data_prepared['positive'] = steam_data_prepared['positive'].astype('int32')
steam_data_prepared['negative'] = steam_data_prepared['negative'].astype('int32')

steam_data_prepared['average_playtime_forever'] = steam_data_prepared['average_playtime_forever'].astype('int32')
steam_data_prepared['average_playtime_2weeks'] = steam_data_prepared['average_playtime_2weeks'].astype('int32')
steam_data_prepared['median_playtime_forever'] = steam_data_prepared['median_playtime_forever'].astype('int32')
steam_data_prepared['median_playtime_2weeks'] = steam_data_prepared['median_playtime_2weeks'].astype('int32')

steam_data_prepared['discount'] = steam_data_prepared['discount'].astype('int32')
steam_data_prepared['peak_ccu'] = steam_data_prepared['peak_ccu'].astype('int32')

# Dejamos tags como objecto tambien

In [11]:
steam_data_prepared.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35860 entries, 0 to 35859
Columns: 484 entries, steam_appid to user_score
dtypes: datetime64[ns](1), float32(1), float64(452), int32(14), int64(1), object(5), string(10)
memory usage: 130.4+ MB


In [12]:
## Cálculo de estadísticas para deducir la unidad de MEDIDA DE TIEMPO

# Seleccionamos las columnas de tiempo de juego
playtime_cols = ['average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks']

# Estadísticas descriptivas principales
print("--- Estadísticas descriptivas de las columnas de tiempo de juego ---")
display(steam_data_prepared[playtime_cols].describe())

# Valores máximos de cada columna
print("\n--- Verificación de máximos absolutos ---")
for col in playtime_cols:
    max_val = steam_data_prepared[col].max()
    max_hours = max_val / 60
    print(f"{col:25} -> Máximo: {max_val:9,} minutos ({max_hours:7,.2f} horas)")

--- Estadísticas descriptivas de las columnas de tiempo de juego ---


,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks
count,3.586000e+04,35860.000000,3.586000e+04,35860.000000
mean,2.585041e+02,14.424902,2.160281e+02,15.039403
std,1.824767e+04,285.777610,1.821954e+04,299.247868
min,0.000000e+00,0.000000,0.000000e+00,0.000000
25%,0.000000e+00,0.000000,0.000000e+00,0.000000
50%,0.000000e+00,0.000000,0.000000e+00,0.000000
75%,0.000000e+00,0.000000,0.000000e+00,0.000000
max,3.429544e+06,20088.000000,3.429544e+06,20088.000000



--- Verificación de máximos absolutos ---
average_playtime_forever  -> Máximo: 3,429,544 minutos (57,159.07 horas)
average_playtime_2weeks   -> Máximo:    20,088 minutos ( 334.80 horas)
median_playtime_forever   -> Máximo: 3,429,544 minutos (57,159.07 horas)
median_playtime_2weeks    -> Máximo:    20,088 minutos ( 334.80 horas)


# Análisis descriptivo de los datos

## Variables String y Object (con PANDAS)



In [13]:
# Seleccionamos las columnas cuyo tipo sea 'string' u 'object'
string_obj_cols = steam_data_prepared.select_dtypes(include=['string', 'object']).columns.tolist()

print("--- Variables de tipo String y Object ---")
print(string_obj_cols)

--- Variables de tipo String y Object ---
['about_the_game', 'categories', 'detailed_description', 'developers', 'estimated_owners', 'full_audio_languages', 'genres', 'name', 'notes', 'packages', 'publishers', 'reviews', 'short_description', 'supported_languages', 'tags']


In [14]:
# 2. Resumen estadístico básico
print("\n--- Resumen General ---")
display(steam_data_prepared[string_obj_cols].describe(include='all'))


--- Resumen General ---


,about_the_game,categories,detailed_description,developers,estimated_owners,full_audio_languages,genres,name,notes,packages,publishers,reviews,short_description,supported_languages,tags
count,33420,35860,33426,35860,35860,35860,35860,35860,6547,35860,35860,3505,33459,35860,11331
unique,33307,5342,33314,25802,14,1520,1629,35731,5880,28385,22972,3474,33247,6693,1
top,Help the beautiful girls to immerse themselves...,"['Single-player', 'Family Sharing']",Help the beautiful girls to immerse themselves...,[],0 - 20000,[],[],Shadow of the Tomb Raider: Definitive Edition,This Game may contain content not appropriate ...,[],[],"“🏆 I loved! ❤️❤️❤️” Player “🏆 Cute kittens, ea...","Collect a 3D puzzle, transferring things to th...",['English'],[]
freq,18,6774,18,2441,22044,21153,2440,7,40,7457,2570,10,17,16165,11331


In [15]:
# 3. Análisis de valores nulos o vacíos
print("\n--- Análisis de valores vacíos y nulos ---")
vacios_df = pd.DataFrame({
    'Valores Nulos (NaN)': steam_data_prepared[string_obj_cols].isna().sum(),
    'Cadenas Vacías ("")': (steam_data_prepared[string_obj_cols] == "").sum(),
    'Colecciones Vacías (listas/diccionarios)': steam_data_prepared[string_obj_cols].apply(
        lambda col: col.apply(lambda x: len(x) == 0 if isinstance(x, (list, dict)) else False)
    ).sum()
})
display(vacios_df)


--- Análisis de valores vacíos y nulos ---


,Valores Nulos (NaN),"Cadenas Vacías ("""")",Colecciones Vacías (listas/diccionarios)
about_the_game,2440,0,0
categories,0,0,0
detailed_description,2434,0,0
developers,0,0,0
estimated_owners,0,0,0
full_audio_languages,0,0,0
genres,0,0,0
name,0,0,0
notes,29313,0,0
packages,0,0,0


In [16]:
# 4. Longitud promedio de los textos descriptivos principales
print("\n--- Estadísticas de longitud de caracteres de los textos ---")
text_cols = ['name', 'detailed_description', 'about_the_game', 'short_description', 'reviews']
for col in text_cols:
    if col in steam_data_prepared.columns:
        lengths = steam_data_prepared[col].fillna("").str.len()
        print(f"{col:25} -> Media: {lengths.mean():7.1f} | Mediana: {lengths.median():7.1f} | Máx: {lengths.max():9,d}")


--- Estadísticas de longitud de caracteres de los textos ---
name                      -> Media:    18.1 | Mediana:    16.0 | Máx:       148
detailed_description      -> Media:  1226.6 | Mediana:  1016.0 | Máx:    60,899
about_the_game            -> Media:  1204.3 | Mediana:  1005.5 | Máx:    60,899
short_description         -> Media:   184.6 | Mediana:   201.0 | Máx:       342
reviews                   -> Media:    33.1 | Mediana:     0.0 | Máx:     2,913


In [17]:
# 5. Top 10 valores más frecuentes para columnas categóricas
print("\n--- Top 10 valores más comunes por categoría ---")
categorical_text_cols = ['developers', 'publishers', 'genres', 'categories']
for col in categorical_text_cols:
    if col in steam_data_prepared.columns:
        print(f"\n[Top 10 en {col.upper()}]")
        print(steam_data_prepared[col].value_counts().head(10))


--- Top 10 valores más comunes por categoría ---

[Top 10 en DEVELOPERS]
developers
[]                                2441
['EroticGamesClub']                 69
['Laush Dmitriy Sergeevich']        50
['Choice of Games']                 48
['Boogygames Studios']              48
['Creobit']                         47
['Bully Revenge Studios']           41
['Cyber Keks']                      38
['Hosted Games']                    35
['KOEI TECMO GAMES CO., LTD.']      35
Name: count, dtype: Int64

[Top 10 en PUBLISHERS]
publishers
[]                       2570
['BFG Entertainment']     173
['8floor']                 90
['EroticGamesClub']        69
['Kagura Games']           66
['Strategy First']         64
['Laush Studio']           61
['HH-Games']               60
['Conglomerate 5']         59
['Hede']                   58
Name: count, dtype: Int64

[Top 10 en GENRES]
genres
[]                                  2440
['Casual', 'Indie']                 1986
['Action', 'Indie']          

## Variables int y float (con POLARS)

In [18]:
import polars as pl

# Definimos las columnas numéricas de interés
numeric_cols = ['required_age', 'price', 'dlc_count', 'metacritic_score', 'achievements',
                'recommendations', 'user_score', 'positive', 'negative',
                'average_playtime_forever', 'average_playtime_2weeks',
                'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu']

# Creamos el DataFrame de Polars
pl_df = pl.from_pandas(steam_data_prepared[numeric_cols])

# Calculamos el resumen descriptivo en Polars
summary_pl = pl_df.describe()

# Redondeo a 3 decimales
summary_pl = summary_pl.with_columns([
    pl.col(c).cast(pl.Float64).round(3) if pl_df[c].dtype.is_numeric() else pl.col(c)
    for c in numeric_cols
])

display(summary_pl.to_pandas())

,statistic,required_age,price,dlc_count,metacritic_score,achievements,recommendations,user_score,positive,negative,average_playtime_forever,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu
0,count,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000,35860.000
1,null_count,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,mean,0.156,4.788,0.503,2.620,18.164,980.420,0.030,1070.200,196.196,258.504,14.425,216.028,15.039,13.128,69.597
3,std,1.596,12.692,7.667,13.815,138.023,19220.153,1.589,20935.715,6582.934,18247.671,285.778,18219.544,299.248,25.989,3964.270
4,min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
5,25%,0.000,0.590,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
6,50%,0.000,2.240,0.000,0.000,2.000,0.000,0.000,5.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000
7,75%,0.000,5.390,0.000,0.000,19.000,0.000,0.000,38.000,11.000,0.000,0.000,0.000,0.000,0.000,0.000
8,max,18.000,999.000,875.000,94.000,5394.000,1753261.000,100.000,2037143.000,1037487.000,3429544.000,20088.000,3429544.000,20088.000,100.000,623941.000


In [19]:
# 2. Métricas adicionales de calidad de datos calculadas puramente en POLARS
total_rows = pl_df.height

# Calculamos nulos, ceros y sus porcentajes utilizando expresiones nativas de Polars
quality_exprs = []
for col in numeric_cols:
    quality_exprs.extend([
        pl.col(col).null_count().alias(f"{col}_nulos"),
        (pl.col(col).null_count() / total_rows * 100).round(2).alias(f"{col}_pct_nulos"),
        (pl.col(col) == 0).sum().alias(f"{col}_ceros"),
        ((pl.col(col) == 0).sum() / total_rows * 100).round(2).alias(f"{col}_pct_ceros")
    ])

# Ejecutamos las agregaciones en un solo paso eficiente
quality_stats = pl_df.select(quality_exprs)

# Reestructuramos el resultado para tenerlo como tabla legible por columna
quality_rows = []
for col in numeric_cols:
    row = {
        "Variable": col,
        "Valores Nulos (NaN)": quality_stats.get_column(f"{col}_nulos")[0],
        "% Nulos": quality_stats.get_column(f"{col}_pct_nulos")[0],
        "Valores Cero (0)": quality_stats.get_column(f"{col}_ceros")[0],
        "% Ceros": quality_stats.get_column(f"{col}_pct_ceros")[0]
    }
    quality_rows.append(row)

quality_metrics_pl = pl.DataFrame(quality_rows)
print("--- Métricas de Calidad de Datos con Polars ---")
display(quality_metrics_pl.to_pandas())

--- Métricas de Calidad de Datos con Polars ---


,Variable,Valores Nulos (NaN),% Nulos,Valores Cero (0),% Ceros
0,required_age,0,0.0,35512,99.03
1,price,0,0.0,7571,21.11
2,dlc_count,0,0.0,30647,85.46
3,metacritic_score,0,0.0,34592,96.46
4,achievements,0,0.0,17418,48.57
5,recommendations,0,0.0,29701,82.82
6,user_score,0,0.0,35847,99.96
7,positive,0,0.0,12225,34.09
8,negative,0,0.0,16070,44.81
9,average_playtime_forever,0,0.0,28233,78.73


In [20]:
# 3. Tabla descriptiva completa unificada
# Transponemos la tabla
summary_long = summary_pl.unpivot(index="statistic", on=numeric_cols)
summary_transposed = summary_long.pivot(on="statistic", index="variable", values="value")
summary_transposed = summary_transposed.rename({"variable": "Variable"})

# Unimos el análisis descriptivo con las métricas de calidad de datos
complete_numeric_analysis_pl = summary_transposed.join(quality_metrics_pl, on="Variable")

print("--- Análisis Descriptivo Completo de Variables Numéricas ---")
display(complete_numeric_analysis_pl.to_pandas())

--- Análisis Descriptivo Completo de Variables Numéricas ---


,Variable,count,null_count,mean,std,min,25%,50%,75%,max,Valores Nulos (NaN),% Nulos,Valores Cero (0),% Ceros
0,required_age,35860.0,0.0,0.156,1.596,0.0,0.00,0.00,0.00,18.0,0,0.0,35512,99.03
1,price,35860.0,0.0,4.788,12.692,0.0,0.59,2.24,5.39,999.0,0,0.0,7571,21.11
2,dlc_count,35860.0,0.0,0.503,7.667,0.0,0.00,0.00,0.00,875.0,0,0.0,30647,85.46
3,metacritic_score,35860.0,0.0,2.620,13.815,0.0,0.00,0.00,0.00,94.0,0,0.0,34592,96.46
4,achievements,35860.0,0.0,18.164,138.023,0.0,0.00,2.00,19.00,5394.0,0,0.0,17418,48.57
5,recommendations,35860.0,0.0,980.420,19220.153,0.0,0.00,0.00,0.00,1753261.0,0,0.0,29701,82.82
6,user_score,35860.0,0.0,0.030,1.589,0.0,0.00,0.00,0.00,100.0,0,0.0,35847,99.96
7,positive,35860.0,0.0,1070.200,20935.715,0.0,0.00,5.00,38.00,2037143.0,0,0.0,12225,34.09
8,negative,35860.0,0.0,196.196,6582.934,0.0,0.00,1.00,11.00,1037487.0,0,0.0,16070,44.81
9,average_playtime_forever,35860.0,0.0,258.504,18247.671,0.0,0.00,0.00,0.00,3429544.0,0,0.0,28233,78.73


#Limpieza de datos

### Demos y Playtests

En Steam se publican, a parte de los juegos acabados, distintas versiones de juegos inacabados, con acceso anticipado y demos. Todas estas versiones tienen unas características diferentes a los lanzamientos de juegos tradicionales y sesgarían nuestro análisis.

In [21]:
# Filtramos y contamos juegos con "Playtest" o "Demo" en el nombre

# Creamos mascaras para filtrar palabras clave
playtest_mask = steam_data_prepared['name'].str.contains('playtest', case=False, na=False)
demo_mask = steam_data_prepared['name'].str.contains('demo', case=False, na=False)
combined_mask = playtest_mask | demo_mask

# Calculamos los totales
playtest_count = playtest_mask.sum()
demo_count = demo_mask.sum()
total_count = combined_mask.sum()

print(f"Juegos que contienen la palabra 'Playtest': {playtest_count}")
print(f"Juegos que contienen la palabra 'Demo': {demo_count}")
print(f"Juegos únicos que contienen 'Playtest' o 'Demo': {total_count}")

Juegos que contienen la palabra 'Playtest': 2358
Juegos que contienen la palabra 'Demo': 154
Juegos únicos que contienen 'Playtest' o 'Demo': 2507


In [22]:
# Creamos el nuevo dataframe filtrando los elementos usando la negación de la máscara combinada anterior
steam_data_noplaytestdemo = steam_data_prepared[~combined_mask].copy()

# Mostramos información sobre el cambio de tamaño para verificar que se han eliminado correctamente
entries_removed = len(steam_data_prepared) - len(steam_data_noplaytestdemo)
print(f"Entradas eliminadas con éxito: {entries_removed}")
print(f"Tamaño del nuevo DataFrame (sin Playtests ni Demos): {steam_data_noplaytestdemo.shape[0]} entradas")


Entradas eliminadas con éxito: 2507
Tamaño del nuevo DataFrame (sin Playtests ni Demos): 33353 entradas


Vamos a comprobar si podemos agrupar algunas entradas duplicadas si tienen el mismo nombre y el mismo desarrollador. Una vez encontrados, nos fijamos tambien en si comparten fecha de lanzamiento u otras variables.

In [23]:
# Comprobando si hay IDs duplicados
print("Indices duplicados:", steam_data_noplaytestdemo.index.duplicated().sum())

Indices duplicados: 0


In [24]:
# Comprobando si hay nombres duplicados que además coincidan en "developers"
# Normalizamos la cadena de texto del nombre
steam_data_noplaytestdemo["name_normalized"] = (steam_data_noplaytestdemo["name"].str.strip().str.casefold())

# Buscamos duplicados que coincidan en las columnas "name_normalized" y "developers"
dupe_nombre_dev = steam_data_noplaytestdemo.duplicated(subset=["name_normalized", "developers"], keep=False)

print("Número de registros duplicados (mismo nombre y mismo desarrollador):", dupe_nombre_dev.sum())


Número de registros duplicados (mismo nombre y mismo desarrollador): 24


In [25]:
# Filtramos el dataframe para mostrar los registros duplicados exactos en nombre y developer
show_dupes = steam_data_noplaytestdemo[dupe_nombre_dev].sort_values(by=["name_normalized", "developers"])

# Mostramos columnas relevantes para los registros duplicados
show_dupes[["name", "release_date", "price", "discount", "estimated_owners", "developers"]]

,name,release_date,price,discount,estimated_owners,developers
2395,Call of Duty®: Modern Warfare® II,2022-10-22,27.990000,60,0 - 20000,"['Infinity Ward', 'Raven Software', 'Beenox', ..."
6467,Call of Duty®: Modern Warfare® II,2022-10-22,27.990000,60,0 - 20000,"['Infinity Ward', 'Raven Software', 'Beenox', ..."
3573,Civilization IV: Beyond the Sword,2007-07-24,5.990000,80,500000 - 1000000,['Firaxis Games']
14106,Civilization IV: Beyond the Sword,2007-07-24,5.990000,0,1000000 - 2000000,['Firaxis Games']
6767,EA SPORTS FC™ 25,2024-09-26,69.989998,0,0 - 20000,"['EA Canada', 'EA Romania']"
20984,EA SPORTS FC™ 25,2024-09-26,69.989998,0,0 - 20000,"['EA Canada', 'EA Romania']"
24652,IL-2 Sturmovik: 1946,2008-06-13,1.990000,80,200000 - 500000,['1C: Maddox Games']
33026,IL-2 Sturmovik: 1946,2008-06-13,1.990000,80,0 - 20000,['1C: Maddox Games']
988,Jagged Alliance 2 Gold,2006-07-06,3.990000,80,50000 - 100000,['Sir-Tech']
27104,Jagged Alliance 2 Gold,2006-07-06,3.990000,80,0 - 20000,['Sir-Tech']


In [26]:
#TODO Hacer funcion para extraer el precio inicial de "packages"

### Variable Objetivo

Steam no proporciona datos precisos de ventas así que necesitaremos definir una variable objectivo sustitutiva que nos sirva de proxy para aproximar el éxito comercial de los lanzamientos de videojuegos.

Como disponemos de un rango de propietarios de los juegos vamos a usar la media aritmética para determinar el punto medio y usar la variable resultante cómo objectivo.

In [26]:
#Formateo de la columna "estimated_owners
# Aplicamos la función para pasar de strings a tuplas con el rango
steam_data_noplaytestdemo['estimated_owners'] = steam_data_noplaytestdemo['estimated_owners'].apply(parse_estimated_owners)

# Aplicamos función lambda con la media aritmética
steam_data_noplaytestdemo['estimated_owners_mid'] = steam_data_noplaytestdemo['estimated_owners'].apply(lambda x: round((x[0] + x[1]) / 2))

# Comprobamos los resultados
display(steam_data_noplaytestdemo[['name', 'estimated_owners', 'estimated_owners_mid']].head())

,name,estimated_owners,estimated_owners_mid
1,Supipara - Chapter 1 Spring Has Come!,"(0, 20000)",10000
2,Mystery Solitaire The Black Raven,"(0, 20000)",10000
3,버튜버 파라노이아 - Vtuber Paranoia,"(0, 20000)",10000
4,Maze Quest VR,"(0, 20000)",10000
5,Agony VR,"(0, 20000)",10000


El dataset utilizado es: "steam_data_noplaytestdemo"

Para preparar el modelo, eliminamos la variable "estimated_owners" del modelo y la definimos como la variable objetivo (y).

La columna "estimated_owner_mid" (cantidad de ventas/descargas) es el valor que quiero predecir.

In [30]:
# Importamos las librerías necesarias
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import numpy as np

# Para evitar errores con columnas de texto restantes (como 'packages' u otras tags no numéricas),
# seleccionamos automáticamente solo las columnas numéricas del dataset.
columnas_numericas = steam_data_noplaytestdemo.select_dtypes(include=[np.number]).columns

# Excluimos la variable objetivo y el ID de Steam
predictores = [col for col in columnas_numericas if col not in ['estimated_owners_mid', 'steam_appid']]
# Rellena los nulos de cada columna numérica con su mediana
for col in predictores:
  [col] = df[col].fillna(df[col].median())
X = steam_data_noplaytestdemo[predictores]
y = steam_data_noplaytestdemo['estimated_owners_mid']

# Dividimos los datos en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, shuffle=True)

# Añadimos la constante para el término de intercepto
X_train = sm.add_constant(X_train)

# Ajustamos el modelo OLS, omitiendo registros que contengan valores nulos (NaNs)
modelo = sm.OLS(endog=y_train, exog=X_train)
modelo = modelo.fit()
print(modelo.summary())

NameError: name 'df' is not defined

In [35]:
# Importamos las librerías necesarias
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import numpy as np
import pandas as pd

# Para evitar errores con columnas de texto restantes (como 'packages' u otras tags no numéricas),
# seleccionamos automáticamente solo las columnas numéricas del dataset.
columnas_numericas = steam_data_noplaytestdemo.select_dtypes(include=[np.number]).columns

# Excluimos la variable objetivo y el ID de Steam
predictores = [col for col in columnas_numericas if col not in ['estimated_owners_mid', 'steam_appid']]

# Rellena los nulos de cada columna numérica con su mediana.
# Si la columna está completamente vacía (mediana es NaN), se rellena con 0.
for col in predictores:
    mediana = steam_data_noplaytestdemo[col].median()
    if pd.isna(mediana):
        steam_data_noplaytestdemo[col] = steam_data_noplaytestdemo[col].fillna(0)
    else:
        steam_data_noplaytestdemo[col] = steam_data_noplaytestdemo[col].fillna(mediana)

X = steam_data_noplaytestdemo[predictores]
y = steam_data_noplaytestdemo['estimated_owners_mid']

# Dividimos los datos en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, shuffle=True)

# Añadimos la constante para el término de intercepto
X_train = sm.add_constant(X_train)

# Ajustamos el modelo OLS, omitiendo registros con nulos residuales si los hubiera
modelo = sm.OLS(endog=y_train, exog=X_train, missing='drop')
modelo = modelo.fit()
print(modelo.summary())

/tmp/ipykernel_5942/1072925981.py:34: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  modelo = modelo.fit()


                             OLS Regression Results                             
Dep. Variable:     estimated_owners_mid   R-squared:                       0.868
Model:                              OLS   Adj. R-squared:                  0.865
Method:                   Least Squares   F-statistic:                     372.0
Date:                  Sun, 22 Nov 2026   Prob (F-statistic):               0.00
Time:                          17:32:13   Log-Likelihood:            -3.8899e+05
No. Observations:                 26682   AIC:                         7.789e+05
Df Residuals:                     26219   BIC:                         7.827e+05
Df Model:                           462                                         
Covariance Type:              nonrobust                                         
                                             coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------

El modelo logra explicar prácticamente el 87% del comportamiento de la variable objetivo (R-squared = 0.868 y Adj. R-squared = 0.865). El R² ajustado es casi idéntico al R² normal, lo que confirma que el alto poder predictivo no es solo producto de haber metido muchas variables, aunque sí hay margen para simplificar.

Al ser el p-valor de F igual a 0.00 (menor a 0.05), (F-statistic) = 0.00, el modelo en su conjunto es estadísticamente significativo. Las variables independientes aportan información real para predecir estimated_owners_mid.

Variables numéricas muy influyentes (P>|t| = 0.000): Las reseñas (positive, negative), las recomendaciones (recommendations), los jugadores concurrentes (peak_ccu) y los tiempos de juego (average_playtime_2weeks, median_playtime_2weeks) tienen un impacto estadísticamente muy fuerte.

Variables que no aportan valor (P>|t| > 0.05): Métricas como achievements (0.544) o metacritic_score (0.331) no tienen efecto significativo en este modelo.



Interpretación de los indicadores finales

Independencia de errores perfecta (Durbin-Watson = 2.001): Es el único indicador sano en este bloque. Al ser prácticamente igual a 2.0, confirma que no hay autocorrelación en los residuos.

Multicolinealidad extrema (Cond. No. = 3.53e+18): Muchas etiquetas tags.* que aparecen juntas casi siempre, o métricas muy correlacionadas como average_playtime y median_playtime, o positive y recommendations. Esto hace que los coeficientes sean inestables.

Residuos muy sesgados y con valores atípicos (Skew = 45.470 y Kurtosis = 3996.501): En una distribución normal ideal, la asimetría (Skew) debería acercarse a 0 y la curtosis (Kurtosis) a 3.
Hay unos pocos juegos con ventas masivas (como Counter-Strike o PUBG) que disparan los errores de predicción y rompen el supuesto de normalidad (Prob(Omnibus) = 0.000 y Prob(JB) = 0.00).

Para corregirlo:

1. Hacer transformación logarítmica (np.log1p) Transformar estimated_owners_mid (y variables muy sesgadas como positive, negative o peak_ccu) reducirá drásticamente el Skew y la Kurtosis, estabilizando las predicciones.
2. Eliminar variables redundantes: Dejar solo con las etiquetas más frecuentes (por ejemplo, las que aparezcan en más de 500 juegos) y eliminar métricas duplicadas (tengo que elegir entre media o mediana de tiempo de juego, no ambas) para bajar el Cond. No..
3. Usa regresión regularizada (Ridge o Lasso): Sustituir OLS por Ridge o Lasso (de sklearn), diseñados específicamente para manejar multicolinealidad y exceso de variables.

In [36]:
# 1. Filtrar nombres de variables con p-valor < 0.05 (excluyendo la constante 'const')
vars_significativas = modelo.pvalues[modelo.pvalues < 0.05].index.tolist()
if "const" in vars_significativas:
  vars_significativas.remove("const")

# 2. Crear el nuevo conjunto X limpio y reentrenar
X_limpio = sm.add_constant(X[vars_significativas])
modelo_limpio = sm.OLS(y, X_limpio).fit()
print(modelo_limpio.summary())

                             OLS Regression Results                             
Dep. Variable:     estimated_owners_mid   R-squared:                       0.893
Model:                              OLS   Adj. R-squared:                  0.893
Method:                   Least Squares   F-statistic:                     1389.
Date:                  Sun, 22 Nov 2026   Prob (F-statistic):               0.00
Time:                          18:32:25   Log-Likelihood:            -4.8627e+05
No. Observations:                 33353   AIC:                         9.729e+05
Df Residuals:                     33152   BIC:                         9.746e+05
Df Model:                           200                                         
Covariance Type:              nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [38]:
#Elimino todas las tags, para que el modelo quede más limpio
X_sin_tags = X.loc[:, ~X.columns.str.startswith("tags.")]

# 2. Aplicamos transformación logarítmica a la variable objetivo (y)
y_log = np.log1p(y)

# 3. Aplicamos logaritmo a las variables numéricas muy sesgadas (con valores >= 0)
cols_sesgadas = [
    "positive",
    "negative",
    "recommendations",
    "peak_ccu",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks",
    "price",
]

for col in cols_sesgadas:
  if col in X_sin_tags.columns:
    X_sin_tags[col] = np.log1p(X_sin_tags[col].clip(lower=0))

# 4. Ajusto el nuevo modelo OLS
X_sin_tags = sm.add_constant(X_sin_tags)
modelo_log = sm.OLS(y_log, X_sin_tags).fit()

# 5. Filtrar automáticamente solo las variables con P < 0.05
vars_sig = modelo_log.pvalues[modelo_log.pvalues < 0.05].index.tolist()
X_final = X_sin_tags[vars_sig]

modelo_final = sm.OLS(y_log, X_final).fit()
print(modelo_final.summary())

                             OLS Regression Results                             
Dep. Variable:     estimated_owners_mid   R-squared:                       0.382
Model:                              OLS   Adj. R-squared:                  0.382
Method:                   Least Squares   F-statistic:                     1872.
Date:                  Sun, 22 Nov 2026   Prob (F-statistic):               0.00
Time:                          18:40:07   Log-Likelihood:                -79362.
No. Observations:                 33353   AIC:                         1.587e+05
Df Residuals:                     33341   BIC:                         1.588e+05
Df Model:                            11                                         
Covariance Type:              nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
cons

/tmp/ipykernel_5942/2557986388.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_sin_tags[col] = np.log1p(X_sin_tags[col].clip(lower=0))
/tmp/ipykernel_5942/2557986388.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_sin_tags[col] = np.log1p(X_sin_tags[col].clip(lower=0))
/tmp/ipykernel_5942/2557986388.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: htt

Baja drásticamente el Cond. No., el Skew y la Kurtosis, logrando un modelo estadísticamente sólido y fácil de interpretar.

El anterior R2 = 0.868 anterior, era un espejismo: En el modelo original (sin logaritmos), 10 juegos con millones de ventas (como Counter-Strike o PUBG) tiraban de la recta de regresión.
El modelo parecía acertar mucho solo porque predecía bien esos pocos extremos, pero fallaba enormemente en el 99% de los juegos normales.


Ahora predice en escala proporcional (porcentajes): Al usar np.log1p, todos los juegos pesan de forma equilibrada. Un R² de 0.382 (explicando casi el 40% de la variación en órdenes de magnitud de ventas usando solo 11 variables) es un resultado lo consideramos un buen resultado.

Pasamos de 26.682 a 33.353 observaciones (No. Observations), lo que hace el análisis más representativo.



Multicolinealidad controlada (Cond. No. = 176 vs 3.53e+18): Pasamos de un colapso numérico total a un valor manejable.

Errores casi normales (Skew = -1.214, Kurtosis = 4.073): Antes tenías una curtosis de 3996.5 (extremos descontrolados); ahora estás muy cerca del ideal teórico (3.0).

Todas las variables son significativas (P>|t| < 0.005): Las 11 variables que quedaron aportan información real, y el estadístico F subió de 372 a 1.872.


Como la variable objetivo y las principales métricas están en logaritmos, los coeficientes funcionan como elasticidades aproximadas (cambio porcentual en propietarios ante un cambio del 1% en la variable):


Las reseñas positivas (positive = +0.8098) y el precio (price = +0.7896) son los mayores impulsores: Por cada 1% que aumentan las reseñas positivas, los propietarios estimados crecen un ~0.81%. El signo positivo del precio indica que los juegos de pago consolidados tienen mayor volumen estimado de base que la enorme masa de juegos gratuitos.



SIGUIENTES PASOS:

- Eliminar una de cada pareja redundante (median_playtime_forever y recommendations).

average_playtime_forever es +0.4567 y median_playtime_forever es -0.4320 (casi el mismo número con signo opuesto). Esto ocurre porque ambas miden lo mismo y se compensan entre sí. Lo mismo pasa con positive (+0.81) y recommendations (-0.24).



- Añadir el Top 20 de etiquetas (tags) más frecuentes y comprobar cuánto mejora el R²

In [39]:
# 1. Separar las columnas de etiquetas (tags) y las métricas base
cols_tags = [col for col in X.columns if col.startswith("tags.")]
X_metricas = X.drop(columns=cols_tags).copy()

# 2. Eliminar variables redundantes que causaban signos invertidos
redundantes = [
    "median_playtime_forever",
    "median_playtime_2weeks",
    "recommendations",
]
X_metricas = X_metricas.drop(
    columns=[c for c in redundantes if c in X_metricas.columns]
)

# 3. Seleccionar el TOP 20 de etiquetas más frecuentes en el dataset
top_20_tags = X[cols_tags].sum().sort_values(ascending=False).head(20).index
X_top_tags = X[top_20_tags].copy()

# 4. Aplicar transformación logarítmica a la variable objetivo (y) y métricas sesgadas
y_log = np.log1p(y)

cols_sesgadas = [
    "positive",
    "negative",
    "peak_ccu",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "price",
]
for col in cols_sesgadas:
  if col in X_metricas.columns:
    X_metricas[col] = np.log1p(X_metricas[col].clip(lower=0))

# 5. Unir métricas limpias + Top 20 etiquetas y añadir constante
X_combinado = pd.concat([X_metricas, X_top_tags], axis=1)
X_combinado = sm.add_constant(X_combinado)

# 6. Ajustar modelo inicial y filtrar automáticamente solo las variables con P < 0.05
modelo_previo = sm.OLS(y_log, X_combinado).fit()
vars_sig = modelo_previo.pvalues[modelo_previo.pvalues < 0.05].index.tolist()

modelo_final = sm.OLS(y_log, X_combinado[vars_sig]).fit()
print(modelo_final.summary())

                                  OLS Regression Results                                 
Dep. Variable:     estimated_owners_mid   R-squared (uncentered):                   0.919
Model:                              OLS   Adj. R-squared (uncentered):              0.919
Method:                   Least Squares   F-statistic:                          3.425e+04
Date:                  Sun, 22 Nov 2026   Prob (F-statistic):                        0.00
Time:                          18:59:08   Log-Likelihood:                         -79707.
No. Observations:                 33353   AIC:                                  1.594e+05
Df Residuals:                     33342   BIC:                                  1.595e+05
Df Model:                            11                                                  
Covariance Type:              nonrobust                                                  
                               coef    std err          t      P>|t|      [0.025      0.975]
-------

• Coeficientes coherentes: Al quitar median_playtime_forever y recommendations, las variables restantes (average_playtime_forever y positive) reflejarán su impacto real sin pelearse entre ellas, bajando aún más el Cond. No..
• Contexto temático sin ruido: El Top 20 de etiquetas (como Indie, Action, Singleplayer, Free to Play, etc.) aporta información clave sobre el género y modelo de negocio del juego sin meter cientos de etiquetas raras que disparan el sobreajuste.
• Poda automática: El paso 6 garantiza que solo entren en el resultado final las métricas y etiquetas del Top 20 que realmente tengan un impacto estadísticamente significativo (P < 0.05).

DAR RESPUESTA A:

• Cuánto ha subido el R-squared al incluir el Top 20 de etiquetas
• Qué géneros o etiquetas tienen mayor peso positivo o negativo en las ventas estimadas


SIGUIENTES ESTUDIOS:

• Qué variables influyen más en la predicción
